In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

In [10]:
DATA_PATH =Path("../outputs/prepared_training_data.csv")
df=pd.read_csv(DATA_PATH)

In [11]:
df["billing_difference"]=(
    df["billed_amount"] - df["expected_total"])

df["billing_ratio"] =np.where(
    df["expected_total"] !=0,
    df["billed_amount"] /df["expected_total"],
    np.nan
)

In [12]:
df["rate_difference"]=(
    df["base_rate_per_mile"] -df["contract_rate_per_mile"]
)

df["rate_ratio"] =np.where(
    df["contract_rate_per_mile"]!=0,
    df["base_rate_per_mile"] /
    df["contract_rate_per_mile"],
    np.nan
)

In [13]:
df["distance_difference"] =(
    df["actual_distance_miles"] -df["typical_distance_miles"]
)
df["distance_ratio"] = np.where(
    df["typical_distance_miles"] !=0,
    df["actual_distance_miles"] /df["typical_distance_miles"],
    np.nan
)

In [14]:
df["calculated_mpg"] =np.where(
    df["fuel_gallons_used"]>0,
    df["actual_distance_miles"] /df["fuel_gallons_used"],
    np.nan
)
df["mpg_difference"] =(
    df["calculated_mpg"]- df["average_mpg"]
)

In [15]:
df["fuel_surcharge_rate_difference"]=(
    df["fuel_surcharge_rate"]-df["contract_fuel_surcharge_rate"]
)
df["fuel_surcharge_rate_ratio"] =np.where(
    df["contract_fuel_surcharge_rate"]!=0,
    df["fuel_surcharge_rate"]/df["contract_fuel_surcharge_rate"],
    np.nan
)

In [16]:
df["chargeable_detention_minutes"]=(
    df["detention_minutes"] -df["detention_free_minutes"]
).clip(lower=0)
df["chargeable_detention_hours"] =(
    df["chargeable_detention_minutes"] / 60
)

In [17]:
df["charge_per_mile"]=np.where(
    df["actual_distance_miles"]> 0,
    df["billed_amount"] /df["actual_distance_miles"],
    np.nan
)

In [18]:
df["charge_per_weight"]= np.where(
    df["weight_lbs"] >0,
    df["billed_amount"] /df["weight_lbs"], np.nan
)

In [19]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [20]:
feature_columns =[
    "billing_difference",
    "billing_ratio",
    "rate_difference",
    "rate_ratio",
    "distance_difference",
    "distance_ratio",
    "fuel_gallons_used",
    "calculated_mpg",
    "mpg_difference",
    "fuel_surcharge_rate_difference",
    "fuel_surcharge_rate_ratio",
    "detention_minutes",
    "chargeable_detention_minutes",
    "charge_per_mile",
    "weight_lbs",
    "pieces"
]

In [21]:
X = df[feature_columns].copy()
X.shape

(85410, 16)

In [22]:
X.isnull().sum()


billing_difference                0
billing_ratio                     0
rate_difference                   0
rate_ratio                        0
distance_difference               0
distance_ratio                    0
fuel_gallons_used                 0
calculated_mpg                    0
mpg_difference                    0
fuel_surcharge_rate_difference    0
fuel_surcharge_rate_ratio         0
detention_minutes                 0
chargeable_detention_minutes      0
charge_per_mile                   0
weight_lbs                        0
pieces                            0
dtype: int64

In [23]:
X = X.fillna(X.median(numeric_only=True))

In [24]:
print("No. of features:", len(feature_columns))
for feature in feature_columns:
    print(feature)

No. of features: 16
billing_difference
billing_ratio
rate_difference
rate_ratio
distance_difference
distance_ratio
fuel_gallons_used
calculated_mpg
mpg_difference
fuel_surcharge_rate_difference
fuel_surcharge_rate_ratio
detention_minutes
chargeable_detention_minutes
charge_per_mile
weight_lbs
pieces


In [25]:
FEATURE_OUTPUT = Path("../outputs/model_features.csv")
X.to_csv(FEATURE_OUTPUT, index=False)
